<div style="background-color:#e6f2ff; padding:20px; border-radius:10px;">
<img style="float:left; margin-right:20px;" src='Figures/alinco.png' width="120"/>
<h1 style="color:#000047;">Actividad 2: PipelineTokenizacionNormalizaciónStemmerLematizacion</h1>
<br style="clear:both"/>
</div>

<div style="border-left:4px solid #000047; padding:10px; margin-top:10px; background:#f5f5f5;">
<b>Objetivo:</b> Crear una clase con un pipeline que tokenice, elimine stopwords, aplique stemming y lematizacion, y mida el impacto en el vocabulario.
</div>

<div style="margin-top:10px;">
<b>Instrucciones generales:</b>
<ul>
<li>Crea tu propia clase para preprocesamiento de texto considerando Normalización de texto, Stemming/Lematización y Tokenización para un corpus en inglés y español</li>
<li>Procesa el corpus provisto de 4 textos en inglés y procesa el corpus de noticias en español</li>
<li>Calcula la reduccion de vocabulario en cada etapa</li>
<li>Visualiza los resultados con matplotlib</li>

</ul>
</div>

In [57]:
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.corpus import stopwords
import string 
import unicodedata
import pandas as pd
import re
class NLPTextCleaningPipeline:
    def __init__(self, idioma = 'spanish', opciones_normalizar = {"urls","correos","menciones","hashtags","numeros","emojis","puntuacion","espacios"},
                 remover_acentos= True, remover_stopwords= True, 
                 normalizar_unicode=True, metodo_stem = None):
        self.idioma = idioma
        self.stopswords = set(stopwords.words('spanish' if idioma=='spanish' else 'english'))
        lenguaje_stem = idioma if idioma in SnowballStemmer.languages else 'english'
        self.stemmer = SnowballStemmer(lenguaje_stem)
        self.lematizer = WordNetLemmatizer()
        self.metodo = metodo_stem
        self.opciones_a_normalizar = opciones_normalizar


    #def tokenizar(self):
    #    pass
   # def stemmeing(self):
    #    pass
   # def Lematizacion(self):
   #     pass
    def Normalizar(self, texto):
        if self.opciones_a_normalizar is None:
            opciones = {"urls", "correos", "menciones", "hashtags",
                    "numeros", "emojis", "puntuacion", "espacios"}
        if "urls"       in self.opciones_a_normalizar: texto = re.sub(r'https?://\S+', '', texto)
        if "correos"    in self.opciones_a_normalizar: texto = re.sub(r'[\w.]+@[\w.]+\.[a-zA-Z]{2,}', '', texto)
        if "menciones"  in self.opciones_a_normalizar: texto = re.sub(r'@\w+', '', texto)
        if "hashtags"   in self.opciones_a_normalizar: texto = re.sub(r'#\w+', '', texto)
        if "numeros"    in self.opciones_a_normalizar: texto = re.sub(r'\S*\d\S*', '', texto)
        if "emojis"     in self.opciones_a_normalizar: texto = re.sub(r'[^\x00-\x7Fáéíóúüñ¡¿ÁÉÍÓÚÜÑa-zA-Z \n.,!?]', '', texto)
        if "puntuacion" in self.opciones_a_normalizar: texto = re.sub(r'[^\w\sáéíóúüñÁÉÍÓÚÜÑ]', ' ', texto, flags=re.UNICODE)
        if "espacios"   in self.opciones_a_normalizar: texto = re.sub(r'\s+', ' ', texto).strip()
        
        return texto.lower()
        
    def quitar_acentos(self, texto):
        """Elimina tildes y diacríticos usando normalización Unicode."""
        nfkd = unicodedata.normalize('NFKD', texto)
        return ''.join(c for c in nfkd if not unicodedata.combining(c))
   # def fit(self):
     #   pass
    #def transform(self):
   #     pass
    def preprocesar(self, texto):
       #1.- Normalizacion
       texto_normalizado = self.Normalizar(texto)
       #2.- Quitar acentos
       texto_normalizado=self.quitar_acentos(texto_normalizado)
       return texto_normalizado

In [58]:
# Corpus simple en inglés
corpus_en = [
    {'Documento':'text_1','texto':'Natural language processing systems are becoming increasingly sophisticated.'},
    {'Documento':'text_2','texto':'Deep learning models have revolutionized text classification and generation tasks.'},
    {'Documento':'text_3','texto':'Tokenization is the fundamental first step in any NLP preprocessing pipeline.'},
    {'Documento':'text_4','texto':'Researchers are studying computational approaches to human language understanding.'},
]
df_corpus_en = pd.DataFrame(corpus_en)
df_corpus_en

,Documento,texto
0,text_1,Natural language processing systems are becomi...
1,text_2,Deep learning models have revolutionized text ...
2,text_3,Tokenization is the fundamental first step in ...
3,text_4,Researchers are studying computational approac...


In [59]:
nlp_cleaning_eng = NLPTextCleaningPipeline(idioma = 'english')

In [60]:
df_corpus_en['texto'][0]

'Natural language processing systems are becoming increasingly sophisticated.'

In [61]:
nlp_cleaning_eng.Normalizar(df_corpus_en['texto'][0])

'natural language processing systems are becoming increasingly sophisticated'

In [62]:
nlp_cleaning_eng.preprocesar(df_corpus_en['texto'][0])

'natural language processing systems are becoming increasingly sophisticated'

In [11]:
#  Corpus de noticias en español
noticias = [
    {"categoria": "tecnología", "texto": "Apple presentó su nuevo chip M4 con IA integrada. La presentación fue en WWDC 2026 #Apple #M4"},
    {"categoria": "tecnología", "texto": "Google lanzó Gemini Ultra, su modelo más poderoso. Supera a GPT-4 en benchmarks https://blog.google.com"},
    {"categoria": "tecnología", "texto": "OpenAI anunció GPT-5 con capacidades multimodales avanzadas @OpenAI"},
    {"categoria": "economía",   "texto": "El peso mexicano se fortalece frente al dólar: $17.50 por USD en mercados internacionales"},
    {"categoria": "economía",   "texto": "La inflación en México cayó al 3.2% en abril 2026, según INEGI #Economia"},
    {"categoria": "economía",   "texto": "Inversión extranjera directa creció 15% en el primer trimestre del año"},
    {"categoria": "ciencia",    "texto": "Científicos descubrieron nueva especie de dinosaurio en Argentina #Paleontología"},
    {"categoria": "ciencia",    "texto": "Investigadores del MIT desarrollaron batería de estado sólido con carga en 5 minutos"},
    {"categoria": "ciencia",    "texto": "La NASA confirmó presencia de agua en la luna sur mediante misión Artemis https://nasa.gov"},
    {"categoria": "salud",      "texto": "Nuevo tratamiento contra el cáncer de páncreas muestra 80% de efectividad en ensayos clínicos"},
    {"categoria": "salud",      "texto": "OMS alerta sobre aumento de casos de resistencia a antibióticos en Latinoamérica #Salud"},
    {"categoria": "salud",      "texto": "Vacuna contra el dengue disponible en farmacias de México a partir de junio 2026"},
]

df_noticias_esp = pd.DataFrame(noticias)
df_noticias_esp

,categoria,texto
0,tecnología,Apple presentó su nuevo chip M4 con IA integra...
1,tecnología,"Google lanzó Gemini Ultra, su modelo más poder..."
2,tecnología,OpenAI anunció GPT-5 con capacidades multimoda...
3,economía,El peso mexicano se fortalece frente al dólar:...
4,economía,La inflación en México cayó al 3.2% en abril 2...
5,economía,Inversión extranjera directa creció 15% en el ...
6,ciencia,Científicos descubrieron nueva especie de dino...
7,ciencia,Investigadores del MIT desarrollaron batería d...
8,ciencia,La NASA confirmó presencia de agua en la luna ...
9,salud,Nuevo tratamiento contra el cáncer de páncreas...
